# Exercise 0 – Summarize job ads

## Setup: modell
Imports, `.env` laddas och modellen skapas mot Nvidias OpenAI-kompatibla API. Nyckeln hämtas från miljövariabeln `NVIDIA_API_KEY`.

In [11]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic import BaseModel, Field
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

model = OpenAIChatModel(
    model_name="nvidia/nemotron-3.5-super-vl-120b-a12b",
    provider = OpenAIProvider(
        base_url = "https://integrate.api.nvidia.com/v1", api_key =os.environ["NVIDIA_API_KEY"]
    ),
)
model

OpenAIChatModel(model_name='nvidia/nemotron-3.5-super-vl-120b-a12b')

## a) Läs in jobbannonserna

- `Path("data").glob("ads*.txt")` hittar alla annonsfiler.
- `sorted()` ger alltid ordningen ads1 → ads2 → ads3.
- `read_text(encoding="utf-8")` läser hela filen som en sträng.
- Resultatet är `ads`, en `list[str]` med 3 annonser.

Annonserna är fritext, inte tabeller, så pandas behövs inte här.

In [12]:
ads = []

for path in sorted(Path("data").glob("ads*.txt")):
    ads.append(path.read_text(encoding="utf-8"))
len(ads)


3

## b) Funktion som sammanfattar en annons

- `job_agent` har instruktioner om vad en bra sammanfattning av en jobbannons ska innehålla.
- `summarize_ad(ad)` tar **en** annons (`str`) och returnerar **en** sammanfattning (`str`).
- `async def` krävs eftersom `agent.run()` är asynkron och måste anropas med `await`.

In [13]:
job_agent = Agent(
    model=model,
    instructions=(
        "You summarize Swedish/English job ads. "
        "Write a short summary: role, company, key requirements and what the job involves."
    ),
)

async def summarize_ad(ad: str) -> str:
    result = await job_agent.run(ad)
    return result.output

## c) Exportera markdown-filer

Loopen går igenom alla annonser:
- `enumerate(ads, start=1)` ger ett nummer per annons, som används i filnamnet (`summary_1.md` …).
- `await summarize_ad(ad)` skickar en annons till agenten och får tillbaka sammanfattningen.
- Varje fil innehåller både originalannonsen och sammanfattningen under egna rubriker.
- `encoding="utf-8"` behövs på Windows för tecken som `→` och emojis.

In [14]:
for i, ad in enumerate(ads, start=1):
    summary = await summarize_ad(ad)                
    with open(f"summary_{i}.md", "w", encoding="utf-8") as file:
        file.write("# Job ad\n\n" + ad + "\n\n# Summary\n\n" + summary)

## e) Structured output med Pydantic

`JobAdSummary` definierar exakt vilka fält agenten ska returnera:
- `job_title`, `description`, `summary`: text
- `responsibilities`: `list[str]`, en lista i stället för en lång sträng
- `words_in_article`: `int`

**`words_in_article` räknas av Python, inte av LLM:en.** LLM:en gissade 830 ord, men det korrekta antalet är 893. Modellen ser tokens och uppskattar, medan `len(ad.split())` räknar exakt. Därför skriver funktionen över LLM:ens värde efter körningen.

`ModelSettings(thinking="low", timeout=60)` minskar modellens interna resonemang och sätter en maxtid per anrop.

In [18]:
from pydantic import BaseModel, Field
from pydantic_ai import ModelSettings

class JobAdSummary(BaseModel):
    job_title: str = Field(description="The job title from the ad")
    description: str = Field(description="1-2 sentences about the company and team")
    summary: str = Field(description="3-5 sentence summary of the whole ad")
    responsibilities: list[str] = Field(description="Main responsibilities as short points")
    words_in_article: int = Field(description="Number of words in the ad")

agent = Agent(
    model=model,
    instructions="You extract structured information from job ads",
    model_settings=ModelSettings(thinking="low", timeout=60),
)

async def summarize(ad: str) -> JobAdSummary:
    result = await agent.run(ad, output_type=JobAdSummary)
    job = result.output
    job.words_in_article = len(ad.split())
    return job

### Felsökning: blev det omförsök?

`len(result.all_messages())` visar hur många meddelanden körningen bestod av. **3** betyder att modellen gav giltig JSON på första försöket: fråga → svar → bekräftelse. Ett högre tal betyder att valideringen misslyckades och modellen fick försöka igen.

Den långa körtiden berodde alltså på svarstiden hos Nvidia, inte på valideringsfel.

In [16]:
result = await agent.run(ads[0], output_type=JobAdSummary)
len(result.all_messages())

3

### LLM vs Python – ordräkning

| Källa | Antal ord |
|---|---|
| LLM (`words_in_article` före fix) | 830 |
| Python (`len(ads[0].split())`) | 893 |

Slutsats: låt LLM:en göra det den är bra på (förstå och sammanfatta) och Python det den är bra på (räkna).

In [17]:
print(job.words_in_article)
print(len(ads[0].split()))

830
893
